In [1]:
from google.cloud import bigquery
from google.oauth2 import service_account

import psycopg2
import pandas as pd
from sqlalchemy import create_engine,URL

In [2]:
credentials = service_account.Credentials.from_service_account_file(
  'c:/Users/Bob/oasisbiz/datawarehouse-390004-34bcb00fb7cb.json'
)
project_id = 'datawarehouse-390004'

In [3]:
client = bigquery.Client(
  project=project_id,
  credentials=credentials
)

In [4]:
# connect to localhost
conn = psycopg2.connect(
  host='localhost',
  port=5432,
  database='postgres',
  user='postgres',
  password='postgres'
)
conn.set_session(autocommit=True)
cursor = conn.cursor()

In [16]:
engine = create_engine(
  URL.create(
    drivername='postgresql+psycopg2',
    host='localhost',
    port=5432,
    database='postgres',
    username='postgres',
    password='postgres'
  )
)

---
x 만들어 올리기

In [ ]:
# 공시지가 - m1.land_price
job = client.query(
  f'''
  select
    pnu,
    base_year,
    amount
  from m1.land_price
  where
    left(pnu,2) = '11' and
    base_year >= '2020'
  '''
)
public_land_price_df = job.result().to_dataframe()

c:\Users\Bob\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\table.py:1957: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [ ]:
try:
  public_land_price_df.to_sql(
    'public_land_price',
    engine,
    if_exists='replace',
    index=False,
  )
except Exception as err:
  print(err)

In [18]:
# 지하철 역 출입구 - m1.subway_mst_xy
job = client.query(
  f'''
  select
    station_nm,
    line_nm,
    ent_num ent_nm,
    lat lon,
    lon lat
  from m1.subway_mst_xy
  where region_gb = '수도권'
  '''
)
subway_ent_df = job.result().to_dataframe()

c:\Users\Bob\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\table.py:1957: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [19]:
subway_ent_df['geom'] = 'POINT (' + subway_ent_df['lon'].astype('string') + ' ' + subway_ent_df['lat'].astype('string') + ')'

In [20]:
try:
  cursor.execute(
    f'''
    create table subway_ent (
      station_nm varchar,
      line_nm varchar,
      ent_nm varchar,
      geom geometry(geometry,4326)
    )
    '''
  )
except Exception as err:
  print(err)

In [23]:
try:
  cursor.execute(
    'delete from subway_ent'
  )
  subway_ent_df[['station_nm','line_nm','ent_nm','geom']].to_sql(
    'subway_ent',
    engine,
    if_exists='append',
    index=False,
  )
except Exception as err:
  print(err)